# L03 · 谱图（spectrogram）直觉——在学 FFT 之前先看结果

**学习目标**
1. 读懂谱图的三个轴（时间 × 频率 × 能量）
2. 在看图中区分纯音（pure tone）、和弦（chord）、扫频（frequency sweep）、噪声
3. 建立「谱图 = Module 5 学完后能亲手算出来的图」的直觉

> ⚠️ 本课**不推导任何公式**，只建立视觉直觉。
> FFT 和 STFT 的数学推导在 L37-L45。

## 1. 什么是谱图？

谱图（Spectrogram）把「声音随时间变化的频率内容」画成一张图：

```
纵轴 ↑  频率 (Hz)
 高频  |  ████░░░░░░░░░░░░░░░░
 中频  |  ░░░░████████░░░░░░░░
 低频  |  ████████████████████
       +────────────────────→  时间 (s)
       颜色深 = 该时刻该频率能量强
```

**三个轴**：
- X 轴：时间（左→右 = 过去→现在）
- Y 轴：频率（下→上 = 低频→高频）
- 颜色：能量（亮/黄 = 强，暗/蓝 = 弱）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 工具：用 matplotlib.pyplot.specgram 显示谱图
# （L43–L45 将从零实现这背后的 STFT 算法；现在只需看图）
#
# 两个参数现在不用理解，L43 会细讲：
#   NFFT=256    每帧取 256 个采样点做 FFT → 频率分辨率 = sr/NFFT = 62.5 Hz/bin
#   noverlap=128  相邻帧重叠 128 点 → 每步跳 128 点（hop）= 8 ms，显示更连续（分辨率由 hop 决定，不是 overlap 本身）
def show_spec(signal, sr, title, ax, ylim=4000):
    ax.specgram(signal, Fs=sr, cmap='inferno', NFFT=256, noverlap=128)
    ax.set_ylim(0, ylim)
    ax.set_xlabel('时间 (s)'); ax.set_ylabel('频率 (Hz)')
    ax.set_title(title)

# L02 里你实现的 make_sine(duration, sr, freq) 封装的正是下面这行 np.sin
# 本课直接用 np.sin 构造信号，让你把注意力放在「看图」上
sr = 16000
t  = np.linspace(0, 2, 2 * sr, endpoint=False)

# 纯音：440 Hz 正弦波
pure_tone = np.sin(2 * np.pi * 440 * t)

fig, ax = plt.subplots(figsize=(8, 3))
show_spec(pure_tone, sr, '纯音 440 Hz（A4）', ax)
plt.tight_layout(); plt.show()
print('观察：一条水平亮线，位于 440 Hz，贯穿整个时间轴。')

### 读图：纯音的特征

- **一条水平亮线** = 整段时间内只有一个频率
- 线的位置 = 频率（440 Hz ≈ Y 轴 440 处）
- 线的亮度 = 能量（全程不变 = 振幅恒定）

你 6 个月后能从这张图的数学原理开始，推导出 `plt.specgram` 内部的每一步计算。

In [ ]:
# 和弦：三个频率同时响
chord = (
    np.sin(2 * np.pi * 261.6 * t) +   # C4
    np.sin(2 * np.pi * 329.6 * t) +   # E4
    np.sin(2 * np.pi * 392.0 * t)     # G4
) / 3.0

fig, ax = plt.subplots(figsize=(8, 3))
show_spec(chord, sr, '和弦 C-E-G（Do-Mi-Sol）', ax)
plt.tight_layout(); plt.show()
print('观察：三条水平亮线，对应 C4=261 Hz, E4=330 Hz, G4=392 Hz。')

In [ ]:
# 扫频（glissando）：频率随时间线性升高
f0, f1 = 200, 3000  # 从 200 Hz 扫到 3000 Hz
sweep = np.sin(2 * np.pi * (f0 * t + (f1 - f0) * t**2 / (2 * t[-1])))  # 积分形式，瞬时频率 f(t)=f0+(f1-f0)t/T

fig, ax = plt.subplots(figsize=(8, 3))
show_spec(sweep, sr, '扫频 200→3000 Hz（glissando）', ax)
plt.tight_layout(); plt.show()
print('观察：一条斜线从低频向高频上升，就像演奏中的滑音。')

In [ ]:
# 白噪声：所有频率同时随机响
rng = np.random.default_rng(42)
noise = rng.standard_normal(len(t))

fig, ax = plt.subplots(figsize=(8, 3))
show_spec(noise, sr, '白噪声（所有频率随机分布）', ax)
plt.tight_layout(); plt.show()
print('观察：整张图均匀地亮——所有频率、所有时刻都有能量。')

In [ ]:
# 四种声音并排对比
signals = [
    (pure_tone, '纯音 440 Hz'),
    (chord,     '和弦 C-E-G'),
    (sweep,     '扫频 200→3000 Hz'),
    (noise,     '白噪声'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, (sig, title) in zip(axes.flat, signals):
    show_spec(sig, sr, title, ax)
plt.suptitle('四种声音的谱图对比', fontsize=13)
plt.tight_layout(); plt.show()

## 2. 规律总结

| 声音类型 | 谱图特征 |
|---|---|
| 纯音（单频率）| 一条水平亮线 |
| 和弦（多频率）| 多条平行水平亮线 |
| 扫频（滑音）  | 一条斜线 |
| 白噪声（white noise）        | 全图均匀亮 |
| 语音          | 水平条纹（共振峰）+ 垂直纹（声道发音节律）|
| 钢琴音符      | 基频（fundamental frequency，F0） + 谐波（整数倍频率，多条等间距亮线）|

## 3. 语音谱图的特征

真实语音比上面复杂：

- **共振峰（Formants）**：声道的谐振频率，形成水平亮带，决定「哪个元音」
- **基频（F0 / Pitch）**：声带振动频率，男声约 100-200 Hz，女声约 180-300 Hz
- **清音/浊音**：浊音（a, e, i）有清晰的谐波结构；清音（s, f）更像噪声

Whisper 输入的就是这种谱图（Mel spectrogram 版本，L46-L47 实现）。

In [ ]:
# 模拟「元音序列」：用不同共振峰组合近似 a / e / i
def vowel_approx(t, f0, f1, f2):
    """用基频 + 两个共振峰合成近似元音。"""
    return (
        0.5 * np.sin(2 * np.pi * f0 * t) +
        0.3 * np.sin(2 * np.pi * f1 * t) +
        0.2 * np.sin(2 * np.pi * f2 * t)
    )

seg_dur = 0.6
sr = 16000
t_seg = np.linspace(0, seg_dur, int(seg_dur * sr), endpoint=False)

# 简化的共振峰参数（非精确语音合成，仅用于谱图演示）
vowel_a = vowel_approx(t_seg, 130,  700, 1200)
vowel_e = vowel_approx(t_seg, 130,  400, 2200)
vowel_i = vowel_approx(t_seg, 130,  300, 3000)
combined = np.concatenate([vowel_a, vowel_e, vowel_i])

fig, ax = plt.subplots(figsize=(9, 3))
show_spec(combined, sr, '模拟元音序列  a → e → i  （共振峰变化）', ax, ylim=4000)
plt.tight_layout(); plt.show()
print('观察：三段声音的高频共振带位置不同，这就是元音区分的关键。')

## ✏️ 练习：预测谱图形状

不运行代码，先用文字描述下面信号的谱图长什么样：

```python
sig_a = np.sin(2*np.pi*1000*t)  # 1000 Hz 纯音
sig_b = sig_a * np.linspace(1, 0, len(t))  # 振幅从 1 线性衰减到 0
sig_c = np.sin(2*np.pi*500*t) + np.sin(2*np.pi*1500*t)  # 两频率和
```

在下面 Markdown cell 里写预测，然后运行代码验证。

**我的预测（运行前填写，不能跳过这步）**

- `sig_a`（1000 Hz 纯音）：_在此写你的预测_
- `sig_b`（1000 Hz 衰减）：_在此写你的预测_
- `sig_c`（500 + 1500 Hz 和弦）：_在此写你的预测_


In [ ]:
sr = 16000
t  = np.linspace(0, 2, 2 * sr, endpoint=False)

sig_a = np.sin(2 * np.pi * 1000 * t)
sig_b = sig_a * np.linspace(1, 0, len(t))
sig_c = np.sin(2 * np.pi * 500 * t) + np.sin(2 * np.pi * 1500 * t)

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, sig, title in zip(axes,
        [sig_a, sig_b, sig_c],
        ['sig_a：1000 Hz', 'sig_b：1000 Hz 衰减', 'sig_c：500 + 1500 Hz']):
    show_spec(sig, sr, title, ax)
plt.tight_layout(); plt.show()

In [ ]:
# 验证：plt.specgram 返回的频率轴确认 440 Hz 纯音峰位于正确 bin
sr = 16000
t  = np.linspace(0, 1, sr, endpoint=False)
pure_tone = np.sin(2 * np.pi * 440 * t)

# specgram 返回 (spectrum, freqs, t_bins, im)
fig, ax = plt.subplots(figsize=(6, 2))
spectrum, freqs, _, _ = ax.specgram(pure_tone, Fs=sr, NFFT=1024)
plt.close()

# 找峰值所在频率 bin
peak_bin = spectrum.mean(axis=1).argmax()
peak_freq = freqs[peak_bin]
print(f'峰值 bin 频率：{peak_freq:.1f} Hz（期望 ≈ 440 Hz）')
assert abs(peak_freq - 440) < 20, f'峰位偏差过大: {peak_freq:.1f} Hz'
print('✅ 440 Hz 纯音的谱图峰值落在正确频率 bin')


## 本课收束

你现在能**看懂**谱图，但还不知道它是**怎么算出来**的。

| 直觉 | 数学 | 课程位置 |
|---|---|---|
| 频率 = Y 轴位置 | DFT 的输出频点 | L37-L39 |
| 能量 = 颜色亮度 | 复数模的平方 | L40-L41 |
| 时间切片 = 窗函数（window function） | STFT 窗函数 | L36, L43-L45 |
| Mel 版谱图 | Mel filterbank | L46-L47 |

Module 5（L32-L53）结束时，你将能从零开始计算本课所有谱图——
包括 `plt.specgram` 内部的每一步，都会是你自己写的代码。

**下一课 L04**：正弦波三要素——频率、振幅、相位各控制什么，
掌握后再去看 L06 的欧拉公式，FFT 就有了具体的「砖头」。